# B — BART-MNLI zero-shot: kiểm số từ scores đã lưu
    **Phạm vi:** lần chạy này tính lại metric trên toàn bộ validation/test từ scores
    NLI lịch sử; không nạp lại BART hoặc chạy mới 28 cặp NLI cho mỗi câu.
    B không fine-tune trọng số bằng GoEmotions. Ngưỡng đã khóa từ validation.
    Mỗi bảng dưới được tính/kiểm khi chạy cell, không chép số dự kiến vào notebook.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/notebook_support.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.notebook_support import artifacts_root
from src.datasets.goemotions import load_goemotions, multi_hot, sha256, REVISION
from src.evaluation.protocols import read_json
from src.evaluation.metrics import evaluate_multilabel
from src.models.transformer import METRIC_NAMES
ARTIFACTS = artifacts_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_colwidth', 100)
print('Python:', sys.version.split()[0])
print('Dataset revision:', REVISION)
print('Artifacts:', ARTIFACTS.name)

## 1. Checkpoint, revision, template và trạng thái run thật

In [ ]:
from src.models.zero_shot import CHECKPOINT, HYPOTHESIS_TEMPLATE
B = ARTIFACTS/'data/processed/zero_shot/full'
metadata = read_json(B/'run_metadata.json')
protocol = read_json(B/'final_protocol.json')
assert metadata['checkpoint'] == CHECKPOINT and metadata['multi_label']
assert metadata['status'] == 'complete' and not metadata['smoke']
display(pd.Series({k: metadata[k] for k in ('checkpoint', 'model_revision',
    'hypothesis_template', 'dtype', 'device', 'sample_count', 'created_at_utc')}))
print('Fine-tune trọng số: KHÔNG. Tìm ngưỡng: đã thực hiện trên validation trước test.')

## 2. Kiểm dataset, checkpoint/protocol và toàn bộ scores
    Hàm dùng chung kiểm hash, mapping và ID; tính lại đủ 72 dòng toàn đồ án.
    Trong notebook này chỉ trình bày sáu dòng của B. Thiếu/sai artifact sẽ dừng.

In [ ]:
from src.notebook_support import benchmark
from src.models.baseline import load_aligned_scores
verified, frames, labels = benchmark(str(ARTIFACTS))
b_rows = pd.DataFrame([r for r in verified['records'] if r['system'] == 'B_bart_mnli'])
assert len(b_rows) == 6
display(b_rows)
print('Sai lệch metric lớn nhất với CSV báo cáo:', verified['metric_max_absolute_difference'])

## 3. Ghép scores theo ID và lấy một ví dụ NLI thật
    HF trả nhãn theo điểm giảm dần. Cell kiểm cách khôi phục đúng thứ tự 28 cột
    bằng một hàng scores thật. Đây là kiểm mapping, không phải suy luận BART mới.

In [ ]:
from src.models.zero_shot import map_pipeline_scores
val = frames['validation']
val_scores = load_aligned_scores(B/'validation_scores.npz', val.id.tolist(), labels)
i = 0
order = np.argsort(-val_scores[i])
hf_order = {'labels': [labels[j] for j in order], 'scores': val_scores[i, order].tolist()}
np.testing.assert_array_equal(map_pipeline_scores(hf_order, labels)[0], val_scores[i])
print('ID:', val.id.iloc[i], '\nVăn bản:', val.text.iloc[i])
print('Nhãn thật:', [labels[j] for j in val.labels.iloc[i]])
display(pd.DataFrame({'label': labels, 'NLI_score': val_scores[i]}).sort_values('NLI_score', ascending=False))

## 4. Tính metric và phân tích nhãn dự đoán quá nhiều
    Scores NLI chưa được hiệu chuẩn thành xác suất cảm xúc. Kết quả B thấp được giữ
    nguyên; không đổi template/ngưỡng theo test để làm đẹp số đã báo cáo.

In [ ]:
metrics = evaluate_multilabel(multi_hot(val.labels.tolist()), val_scores, labels, 0.5)
np.testing.assert_allclose([metrics[k] for k in METRIC_NAMES],
                           [metadata['metrics'][k] for k in METRIC_NAMES], atol=1e-12, rtol=0)
display(pd.Series({k: metrics[k] for k in (*METRIC_NAMES, 'mean_predicted_labels', 'empty_prediction_count')}))
display(pd.DataFrame(metrics['per_label'])[['label', 'support', 'predicted_support', 'precision', 'recall', 'f1']])

## 5. So sánh ba ngưỡng đã khóa trên validation và test

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, metric in zip(axes, ('macro_f1', 'micro_f1')):
    b_rows.pivot(index='threshold_mode', columns='split', values=metric).plot.bar(ax=ax)
    ax.set_title(metric); ax.set_ylim(0, 1); ax.set_xlabel('Ngưỡng đã khóa')
plt.tight_layout(); plt.show(); plt.close()
print('B chỉ có một run; không tạo mean±std từ một run.')

## 6. Cách chạy lại suy luận NLI khi cần
    Code thực thi nằm tại `scripts/zero_shot/run_zero_shot.py`: nạp pretrained,
    tạo 28 giả thuyết, dự đoán `multi_label=True`, ghép nhãn và lưu từng batch.
    Một lượt full trước đây mất khoảng 19 phút cho validation trên máy chạy.
    Notebook đã chạy này dùng lại scores đó và đã đối chiếu đủ sáu dòng B.

In [ ]:
print('Đã tính lại B từ scores lịch sử:', len(b_rows), 'dòng; số mẫu theo split:')
display(b_rows.groupby('split')['n_samples'].first())
print('NLI inference mới trong notebook này: KHÔNG; huấn luyện lại: KHÔNG.')